# 00 — Setup & smoke test
Verifies the environment, runs the **demo-scale** pipeline end-to-end with the MockLLM
(integration test only — **never paper numbers**), and demonstrates checkpoint/resume.

**Checkpoint levels:** (1) round-level `.ckpt` files in `checkpoints/<run_id>/`,
(2) run-level `results/registry.json` (completed runs are skipped),
(3) HF `checkpoint-*` dirs for QLoRA/classifier training.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
print("project root:", ROOT)

project root: /home/raad/Papers/socialLLMs/traitmix


In [10]:
# 1) Install core requirements (safe to re-run)
!pip install -r ../requirements.txt

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## vLLM server (on your 24 GB machine — run in a separate terminal)
```bash
pip install vllm==0.5.4
# Primary model (E1-E3, E5 non-swap):
vllm serve meta-llama/Llama-3.1-8B-Instruct --max-model-len 4096 --gpu-memory-utilization 0.90
# After notebook 03 (T-arm):
vllm serve meta-llama/Llama-3.1-8B-Instruct --enable-lora --lora-modules big5=checkpoints/qlora_big5chat/final
# Model swaps for E1_QWEN / E5: restart with Qwen/Qwen2.5-14B-Instruct-AWQ or google/gemma-2-9b-it
```

In [5]:
# 2) Smoke run (MockLLM, N=8, T=4) — DEMO-SCALE ONLY
from traitmix.utils import load_config, Registry
from traitmix.runner import run_one
cfg = load_config(ROOT / "configs" / "smoke.yaml")
row = run_one(cfg, seed=1, force=True, keep_ckpt=True)
{k: v for k, v in row.items() if k in ("run_id", "pol_var", "CI_mean_gain_vs_pre", "trait_drift_mean", "runtime_s")}

{'run_id': 'smoke_b76bfedcf9_s1',
 'CI_mean_gain_vs_pre': -0.0012488920436751296,
 'trait_drift_mean': 0.143964645841715,
 'runtime_s': 0.4}

In [6]:
# 3) Checkpoint/resume demonstration: wipe the registry entry, keep round ckpts, rerun.
# The engine resumes from the last saved round instead of starting over.
from traitmix.utils import run_id, Registry
from traitmix import checkpointing as ck
rid = run_id(cfg, 1); reg = Registry(); reg.mark(rid, status="interrupted")
print("latest checkpoint round:", (ck.latest(rid) or {}).get("t_done"))
row2 = run_one(cfg, seed=1, resume=True, force=True)
print("resumed & completed:", row2["run_id"], "runtime_s:", row2["runtime_s"])

latest checkpoint round: 4
resumed & completed: smoke_b76bfedcf9_s1 runtime_s: 0.0


In [7]:
# 4) Sanity: raw_results has rows; registry marks them done
from traitmix.utils import read_rows
df = read_rows()
df.tail(3)[[c for c in ["run_id", "config", "seed", "backend", "runtime_s"] if c in df]]

,run_id,config,seed,backend,runtime_s
0,smoke_b76bfedcf9_s1,smoke,1,mock,0.4
1,smoke_b76bfedcf9_s1,smoke,1,mock,0.0


**Gate to proceed:** both cells above completed and `backend == mock` rows appear.
Real experiments (notebooks 02+) require the vLLM server and will refuse mock-only shortcuts.